# Notebook Setup

## Setting Directories

In [1]:
import sys
import os
from pathlib import Path

# Path to your project root
project_dir = Path(r"C:\Users\dmika\DEV\Projects-local\dp100-learn")

# Change the working directory
os.chdir(project_dir)

# Add to sys.path if not already there
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

## Imports

### Local Imports

In [2]:
# Now you can import from utils
from utils.consts import SUBSCRIPTION_ID, PREFERED_RESOURCE_LOCATION, MAIN_STORAGE_ACCOUNT_ACCESS_KEY

### General Imports

In [3]:
import numpy as np
import pandas as pd

### Azure Imports

In [4]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient

## Consts

In [5]:
subscription_id = SUBSCRIPTION_ID
azure_credentials = DefaultAzureCredential()

## Other

In [6]:
resource_group_name = "ml-workspace-dev"
resource_group_location = PREFERED_RESOURCE_LOCATION

In [7]:
storage_account_name = "dmpdp100storageaccount99"  # must be globally unique
storage_account_location = PREFERED_RESOURCE_LOCATION
storage_container_name = "dmpdp100data"
storage_account_access_key = MAIN_STORAGE_ACCOUNT_ACCESS_KEY

In [8]:
azureml_workspace_name = "mlw-dp100-labs"
azureml_resource_location = PREFERED_RESOURCE_LOCATION

In [9]:
datastore_name = "dmdp100datastore"

In [10]:
ml_client = MLClient(
    credential=azure_credentials,
    subscription_id=subscription_id,
    resource_group_name=resource_group_name,
    workspace_name=azureml_workspace_name
)

# Setup a Resource Group

In [11]:
from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.resource import ResourceManagementClient

# Create resource management client
resource_client = ResourceManagementClient(azure_credentials, subscription_id)
try:
    # Try to get existing resource group
    rg_result = resource_client.resource_groups.get(resource_group_name)
    print(f"Resource group '{rg_result.name}' already exists in region '{rg_result.location}'")
except ResourceNotFoundError:
    # Create resource group if it doesn't exist
    rg_result = resource_client.resource_groups.create_or_update(
        resource_group_name,
        {"location": resource_group_location}
    )
    print(f"Provisioned resource group '{rg_result.name}' in the {rg_result.location} region")


# Optional lines to delete the resource group. begin_delete is asynchronous.
# poller = resource_client.resource_groups.begin_delete(rg_result.name)
# result = poller.result()

ModuleNotFoundError: No module named 'azure.mgmt.resource'

# Setup Storage Account

## Create a Storage Account

In [12]:
from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.storage import StorageManagementClient

storage_client = StorageManagementClient(azure_credentials, subscription_id)

try:
    # Try to get existing storage account
    storage_account = storage_client.storage_accounts.get_properties(
        resource_group_name=resource_group_name,
        account_name=storage_account_name
    )
    print(f"Storage account already exists: {storage_account.name}")
except ResourceNotFoundError:
    # If not found, create new storage account
    print("Creating storage account...")
    poller = storage_client.storage_accounts.begin_create(
        resource_group_name=resource_group_name,
        account_name=storage_account_name,
        parameters={
            "location": storage_account_location,
            "sku": {"name": "Standard_LRS"},
            "kind": "StorageV2",
            "enable_https_traffic_only": True
        }
    )
    storage_account = poller.result()
    print(f"Storage account created: {storage_account.name}")

# Extract the resource ID
storage_resource_id = storage_account.id
print(f"Storage Resource ID: {storage_resource_id}")

Storage account already exists: dmpdp100storageaccount99
Storage Resource ID: /subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.Storage/storageAccounts/dmpdp100storageaccount99


## Create Data Container

In [13]:
from azure.storage.blob import BlobServiceClient

# Build connection string
connection_string = (
    f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};"
    f"AccountKey={storage_account_access_key};EndpointSuffix=core.windows.net"
)

# Create blob service client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Create container (if not exists)
try:
    blob_service_client.create_container(storage_container_name)
    print(f"Container '{storage_container_name}' created.")
except Exception as e:
    if "ContainerAlreadyExists" in str(e):
        print(f"Container '{storage_container_name}' already exists.")
    else:
        raise

Container 'dmpdp100data' already exists.


# Setup Azure ML Workspace

## Creating the Workspace

In [22]:
from azure.core.exceptions import ResourceNotFoundError
from azure.ai.ml.entities import Workspace

try:
    # Try to get existing workspace
    ws = ml_client.workspaces.get(azureml_workspace_name)
    print(f"AML workspace already exists: {ws.name}")
except (ResourceNotFoundError, TypeError):
    ml_client = MLClient(
        credential=azure_credentials,
        subscription_id=subscription_id,
        resource_group_name=resource_group_name,
    )
    # Create new AML workspace
    ws = Workspace(
        name=azureml_workspace_name,
        location=azureml_resource_location,
        storage_account=storage_resource_id
    )
    print("Creating AML workspace...")
    ml_client.workspaces.begin_create(ws).result()
    print(f"Workspace '{azureml_workspace_name}' created with default storage: {storage_account_name}")
    ml_client = MLClient(
        credential=azure_credentials,
        subscription_id=subscription_id,
        resource_group_name=resource_group_name,
        workspace_name=azureml_workspace_name
    )


Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Creating AML workspace...


The deployment request mlw-dp100-labs-4713501 was accepted. ARM deployment URI for reference: 
https://portal.azure.com//#blade/HubsExtension/DeploymentDetailsBlade/overview/id/%2Fsubscriptions%2Fa1267753-4c98-48c1-a8e9-9c7169202ffd%2FresourceGroups%2Fml-workspace-dev%2Fproviders%2FMicrosoft.Resources%2Fdeployments%2Fmlw-dp100-labs-4713501
Creating Key Vault: (mlwdp100keyvault849a59e3  ) ..  Done (18s)
Creating Log Analytics Workspace: (mlwdp100logalytidde834df  )   Done (21s)
Creating AzureML Workspace: (mlw-dp100-labs  ) ...  Done (24s)
Creating Application Insights: (mlwdp100insights097d4f78  )  Done (24s)
Total time : 50s



Workspace 'mlw-dp100-labs' created with default storage: dmpdp100storageaccount99


## Setup the Data

### Create a DataStore

In [25]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration

store = AzureBlobDatastore(
    name=datastore_name,
    description="Blob Storage for DP-100 certification prep",
    account_name=storage_account_name,
    container_name=storage_container_name, 
    credentials=AccountKeyConfiguration(
        account_key=storage_account_access_key
    ),
    type="azure_blob",
)

ml_client.create_or_update(store)

AzureBlobDatastore({'type': <DatastoreType.AZURE_BLOB: 'AzureBlob'>, 'name': 'dmdp100datastore', 'description': 'Blob Storage for DP-100 certification prep', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/datastores/dmdp100datastore', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x00000257F1140110>, 'credentials': {'type': 'account_key'}, 'container_name': 'dmpdp100data', 'account_name': 'dmpdp100storageaccount99', 'endpoint': 'core.windows.net', 'protocol': 'https'})

In [26]:
stores = ml_client.datastores.list()
for ds_name in stores:
    print(ds_name.name)

dmdp100datastore
workspacefilestore
workspaceblobstore
workspaceworkingdirectory
workspaceartifactstore


### Create Data Assets

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

file_path = './data/telco-churn-data/telco-customer-churn.csv'

my_data = Data(
    path=file_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FILE,
    description="Data asset pointing to a local file, automatically uploaded to the default datastore",
    name="telco-churn-file"
)

ml_client.data.create_or_update(my_data)

Uploading telco-customer-churn.csv (< 1 MB): 100%|##########| 970k/970k [00:00<00:00, 3.95MB/s]




Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/980708968c362ca31f4225d8576b9cab/telco-customer-churn.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-file', 'description': 'Data asset pointing to a local file, automatically uploaded to the default datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-file/versions/1', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x00000257F152C260>, 's

In [12]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

folder_path = './data/telco-churn-data'

my_data = Data(
    path=folder_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FOLDER,
    description="Data asset pointing to data-asset-path folder in datastore",
    name="telco-churn-folder",
)

ml_client.data.create_or_update(my_data)

Uploading telco-churn-data (0.97 MBs): 100%|##########| 970617/970617 [00:00<00:00, 2793185.83it/s]




Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/77b974231da746cacc3c272c0aca25d55dcb1bf37567612f7732bf8ac0df1744/telco-churn-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_folder', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-folder', 'description': 'Data asset pointing to data-asset-path folder in datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-folder/versions/2', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x000001A8F940EDD0

In [16]:
%%writefile data/telco-churn-data/MLTable

paths:
  - file: ./telco-customer-churn.csv
transformations:
  - read_delimited:
        delimiter: ','
        encoding: 'ascii'

Overwriting data/telco-churn-data/MLTable


In [17]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_path = './data/telco-churn-data'

my_data = Data(
    path=data_path,
    type=AssetTypes.MLTABLE,
    description="MLTable pointing to telco-customer-churn.csv in data folder",
    name="telco-churn-table",
    datastore=datastore_name,

)

ml_client.data.create_or_update(my_data)

Uploading telco-churn-data (0.97 MBs): 100%|##########| 970595/970595 [00:00<00:00, 2842882.30it/s]




Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/67bbe0434ee4e9154f85f403f18de4e49ef43c66b5844f85418a3218b89d8b33/telco-churn-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['./telco-customer-churn.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-table', 'description': 'MLTable pointing to telco-customer-churn.csv in data folder', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-table/versions/2', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object

### Read the Data Assets Locally

In [11]:
data_asset = ml_client.data.get("telco-churn-file", version="1")
df = pd.read_csv(data_asset.path)
df.sample(2)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
6959,1024-KPRBB,Female,0,No,No,38,Yes,No,Fiber optic,No,...,No,No,Yes,Yes,One year,Yes,Mailed check,89.1,3342,No
840,0727-BMPLR,Female,1,No,No,55,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Electronic check,100.0,5509.3,Yes


In [ ]:
# import mltable

data_asset = ml_client.data.get("telco-churn-folder", version="1")
path = {
  'folder': data_asset.path
}
# tbl = mltable.from_delimited_files(paths=[path])
# df = tbl.to_pandas_dataframe()
df = pd.read_csv(os.path.join(data_asset.path, 'telco-customer-churn.csv'))
df.sample(2)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
4757,1582-RAFML,Male,0,No,No,1,Yes,Yes,DSL,No,...,No,Yes,No,No,Month-to-month,No,Mailed check,60.1,60.1,Yes
3583,2292-XQWSV,Male,0,Yes,Yes,40,No,No phone service,DSL,No,...,Yes,Yes,Yes,Yes,One year,No,Mailed check,60.3,2448.5,No


In [18]:
import mltable

data_asset = ml_client.data.get("telco-churn-table", version="2")

tbl = mltable.load(f"azureml:/{data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,False,True,False,1,False,No phone service,DSL,No,...,No,No,No,No,Month-to-month,True,Electronic check,29.85,29.85,False
1,5575-GNVDE,Male,False,False,False,34,True,No,DSL,Yes,...,Yes,No,No,No,One year,False,Mailed check,56.95,1889.50,False
2,3668-QPYBK,Male,False,False,False,2,True,No,DSL,Yes,...,No,No,No,No,Month-to-month,True,Mailed check,53.85,108.15,True
3,7795-CFOCW,Male,False,False,False,45,False,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,False,Bank transfer (automatic),42.30,1840.75,False
4,9237-HQITU,Female,False,False,False,2,True,No,Fiber optic,No,...,No,No,No,No,Month-to-month,True,Electronic check,70.70,151.65,True


In [12]:
import mltable

registered_data_asset = ml_client.data.get(name='diabetes-table2', version=1)
tbl = mltable.load(f"azureml:/{registered_data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,PatientID,Pregnancies,PlasmaGlucose,DiastolicBloodPressure,TricepsThickness,SerumInsulin,BMI,DiabetesPedigree,Age,Diabetic
0,1354778,0,171,80,34,23,43.509726,1.213191,21,False
1,1147438,8,92,93,47,36,21.240576,0.158365,23,False
2,1640031,7,115,47,52,35,41.511523,0.079019,23,False
3,1883350,9,103,78,25,304,29.582192,1.282870,43,True
4,1424119,1,85,59,27,35,42.604536,0.549542,22,False


In [50]:
data_asset = ml_client.data.get(name="telco-churn-table", version=1)

# create a table
tbl = mltable.load(f"azureml:/{data_asset.id}")

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


## Setup an Environment

### Setup the Compute

### Compute Cluster

### Compute Instance

In [35]:
# Compute Instances need to have a unique name across the region.
# Here we create a unique name with current datetime
from azure.ai.ml.entities import ComputeInstance
import datetime

ci_basic_name = "dp100ci" + datetime.datetime.now().strftime("%Y%m%d%H%M")
ci_basic_name = ci_basic_name[:24]
ci_basic = ComputeInstance(name=ci_basic_name, size="STANDARD_DS11_V2", idle_time_before_shutdown_minutes="15")
ml_client.begin_create_or_update(ci_basic).result()

ComputeInstance({'state': 'Running', 'last_operation': {'operation_name': 'Create', 'operation_time': '2025-09-10T17:21:46.731Z', 'operation_status': 'Succeeded', 'operation_trigger': 'User'}, 'os_image_metadata': <azure.ai.ml.entities._compute._image_metadata.ImageMetadata object at 0x00000257F19429C0>, 'services': [{'display_name': 'Jupyter', 'endpoint_uri': 'https://dp100ci202509101921.westeurope.instances.azureml.ms/tree/'}, {'display_name': 'Jupyter Lab', 'endpoint_uri': 'https://dp100ci202509101921.westeurope.instances.azureml.ms/lab'}], 'type': 'computeinstance', 'created_on': '2025-09-10T17:21:33.372399+0000', 'provisioning_state': 'Succeeded', 'provisioning_errors': None, 'name': 'dp100ci202509101921', 'description': None, 'tags': None, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/computes/dp100ci202509101921', 'Resource_

### Create a datastore